# Collocation

In [1]:
import polars as pl
import polars_corpus as plc
import math

In [2]:
bnc = pl.read_parquet("bnc.parquet").filter(pl.col('mode')=='written')

In [3]:
m = plc.search_cqp(bnc, '[token="dangerous"]')

In [7]:
m.concordance(window=3)

token_left_context,token,token_right_context
list[str],list[str],list[str]
"[""it"", ""was"", ""too""]","[""dangerous""]","[""."", ""One"", ""of""]"
"[""away"", ""could"", ""be""]","[""dangerous""]","[""and"", ""could"", ""therefore""]"
"[""behind"", ""in"", ""the""]","[""dangerous""]","[""town"", ""at"", ""the""]"
"[""would"", ""be"", ""‘""]","[""dangerous""]","[""’"", ""for"", ""the""]"
"[""set"", ""of"", ""potentially""]","[""dangerous""]","[""encounters"", ""between"", ""the""]"
…,…,…
"[""as"", ""intended"", "",""]","[""dangerous""]","["","", ""exhilarating"", ""and""]"
"[""a"", ""dark"", ""and""]","[""dangerous""]","[""love"", ""story"", "",""]"
"[""where"", ""the"", ""most""]","[""dangerous""]","[""place"", ""in"", ""the""]"


In [14]:
 m.collocates("token", window=3)

collocate,freqs
str,struct[4]
"""after""","{19,30564,87497,100204318}"
"""know""","{25,30564,60728,100204318}"
"""Too""","{5,30564,1622,100204318}"
"""where""","{20,30564,80070,100204318}"
"""women""","{13,30564,31901,100204318}"
…,…
"""new""","{12,30564,94865,100204318}"
"""playing""","{12,30564,9132,100204318}"
"""likely""","{7,30564,21965,100204318}"


In [15]:
collocs = m.collocates("token", window=3).sort(
    by=pl.col("freqs").struct.field("f12"), descending=True
)
collocs

collocate,freqs
str,struct[4]
""".""","{1648,30564,4125266,100204318}"
""",""","{1416,30564,4433654,100204318}"
"""and""","{1180,30564,2283660,100204318}"
"""a""","{1041,30564,1839904,100204318}"
"""the""","{1027,30564,5012766,100204318}"
…,…
"""quality""","{5,30564,14058,100204318}"
"""black""","{5,30564,17668,100204318}"
"""second""","{5,30564,32802,100204318}"


In [10]:
ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(20)
)
ll

collocate,freqs,LL
str,struct[4],f64
"""potentially""","{150,30564,2264,100204318}",1326.453024
"""most""","{245,30564,83291,100204318}",673.463291
"""be""","{576,30564,587749,100204318}",556.628729
"""very""","{219,30564,91182,100204318}",523.083232
"""is""","{709,30564,873508,100204318}",509.426343
…,…,…
"""are""","{304,30564,410030,100204318}",183.28141
"""precedent""","{24,30564,674,100204318}",181.758482
"""situation""","{52,30564,13808,100204318}",166.055706


In [7]:
# double checking LL for 'potentially dangerous'

# a = frequency of node-collocate pairs
a = 154
# b = frequency of node without collocate
b = 32676 - a
# c = frequency of collocate without node
c = 2373 - a

# d = words in corpus - occurrences of node and collocate
N = 112429158
d = N - a - b - c

LL2 = 2 * (
    a * math.log(a)
    + b * math.log(b)
    + c * math.log(c)
    + d * math.log(d)
    - (a + b) * math.log(a + b)
    - (a + c) * math.log(a + c)
    - (b + d) * math.log(b + d)
    - (c + d) * math.log(c + d)
    + (N) * math.log(N)
)
LL2

1370.0401501655579

In [16]:
collocs = m.collocates(pl.struct(pl.col("token"),pl.col("pos")), window=3)

In [20]:
ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(100)
)
ll

collocate,freqs,LL
struct[2],struct[4],f64
"{""potentially"",""ADV""}","{150,30564,2264,100204318}",1326.453024
"{""most"",""ADV""}","{241,30564,53455,100204318}",851.405169
"{""more"",""ADV""}","{288,30564,129085,100204318}",651.42637
"{""be"",""VERB""}","{576,30564,587747,100204318}",556.631446
"{""very"",""ADV""}","{218,30564,85499,100204318}",543.59076
…,…,…
"{""risky"",""ADJ""}","{6,30564,595,100204318}",30.401283
"{""foolish"",""ADJ""}","{7,30564,1014,100204318}",30.335837
"{""subversive"",""ADJ""}","{5,30564,306,100204318}",30.076549


In [21]:
ll.filter(pl.col('collocate').struct.field("pos")=="ADV")

collocate,freqs,LL
struct[2],struct[4],f64
"{""potentially"",""ADV""}","{150,30564,2264,100204318}",1326.453024
"{""most"",""ADV""}","{241,30564,53455,100204318}",851.405169
"{""more"",""ADV""}","{288,30564,129085,100204318}",651.42637
"{""very"",""ADV""}","{218,30564,85499,100204318}",543.59076
"{""too"",""ADV""}","{171,30564,58673,100204318}",466.880134
…,…,…
"{""possibly"",""ADV""}","{18,30564,5732,100204318}",51.492538
"{""inherently"",""ADV""}","{8,30564,399,100204318}",51.372056
"{""sometimes"",""ADV""}","{21,30564,13913,100204318}",33.678448


In [22]:
ll.filter(pl.col('collocate').struct.field("pos")=="ADJ")

collocate,freqs,LL
struct[2],struct[4],f64
"{""driving"",""ADJ""}","{33,30564,1058,100204318}",241.116546
"{""difficult"",""ADJ""}","{38,30564,19361,100204318}",77.388093
"{""dangerous"",""ADJ""}","{20,30564,5093,100204318}",65.395101
"{""violent"",""ADJ""}","{12,30564,2495,100204318}",43.76904
"{""unpredictable"",""ADJ""}","{8,30564,649,100204318}",43.678545
…,…,…
"{""exciting"",""ADJ""}","{11,30564,2898,100204318}",35.274374
"{""unhealthy"",""ADJ""}","{5,30564,260,100204318}",31.692151
"{""risky"",""ADJ""}","{6,30564,595,100204318}",30.401283


In [23]:
ll.filter(pl.col('collocate').struct.field("pos")=="SUBST")

collocate,freqs,LL
struct[2],struct[4],f64
"{""substances"",""SUBST""}","{37,30564,1254,100204318}",266.214656
"{""precedent"",""SUBST""}","{24,30564,674,100204318}",181.758482
"{""situation"",""SUBST""}","{52,30564,13808,100204318}",166.055706
"{""driving"",""SUBST""}","{25,30564,1124,100204318}",165.719177
"{""chemicals"",""SUBST""}","{25,30564,1735,100204318}",144.187736
…,…,…
"{""waters"",""SUBST""}","{10,30564,2060,100204318}",36.647608
"{""criminals"",""SUBST""}","{7,30564,794,100204318}",33.639145
"{""species"",""SUBST""}","{16,30564,8693,100204318}",30.848059


In [40]:
m = plc.search_cqp(bnc, '[token="time"]')
collocs = m.collocates('lemma', window=3)
time_ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(200)
)

In [39]:
m = plc.search_cqp(bnc, '[token="money"]')
collocs = m.collocates('lemma', window=3)
money_ll = (
    collocs.with_columns(LL=pl.col("freqs").corpus.loglik())
    .sort(by="LL", descending=True)
    .head(200)
)

In [41]:
time_ll.join(money_ll, on='collocate', how='full').drop_nulls().head(10)

collocate,freqs,LL,collocate_right,freqs_right,LL_right
str,struct[4],f64,str,struct[4],f64
"""spend""","{2854,782610,19975,100204318}",11589.30049,"""spend""","{1091,171546,19975,100204318}",5505.626326
"""for""","{18661,782610,811985,100204318}",16029.980278,"""for""","{4108,171546,811985,100204318}",3519.626653
"""save""","{390,782610,10847,100204318}",589.227317,"""save""","{494,171546,10847,100204318}",2313.281791
"""amount""","{504,782610,18008,100204318}",567.447713,"""amount""","{526,171546,18008,100204318}",2009.218497
"""lot""","{551,782610,16691,100204318}",758.191075,"""lot""","{503,171546,16691,100204318}",1951.391578
"""enough""","{399,782610,27367,100204318}",128.903261,"""enough""","{430,171546,27367,100204318}",1146.413502
"""get""","{1525,782610,118045,100204318}",332.420691,"""get""","{838,171546,118045,100204318}",1117.776869
"""to""","{20978,782610,2360973,100204318}",345.700065,"""to""","{6232,171546,2360973,100204318}",1047.342774
"""more""","{1841,782610,192735,100204318}",70.588629,"""more""","{1037,171546,192735,100204318}",966.441662
